# Welcome! Let's Train Your Gesture Model

Welcome to the **HELLO, VISION!** hands-on training notebook. Here, you will turn the hand gesture samples you recorded in the web app into your own working **K-Nearest Neighbours (KNN)** gesture classifier.

### Before you start
Make sure you already downloaded **`samples.json`** from the **HELLO, VISION!** app:
1. Open **Custom Gestures -> 1. My Gestures**.
2. Collect around **30 samples** for each gesture you want the computer to learn.
3. Click **Download Dataset (.json)** to save **`samples.json`** on your computer.

### How to use this notebook
- Click **Runtime -> Run all** at the top (or press the Play button beside each code cell from top to bottom).
- You only need to interact with **Step 1** (when Colab asks you to upload **`samples.json`**) and **Step 8** (to receive your downloaded **`knn-gesture-model.json`**). Every other cell runs automatically!

## Step 1 – Upload Your Gesture Samples (`samples.json`)

**What we're doing:** Loading the hand gesture examples you collected into Google Colab.

**What you need to do:** Run the code cell below. When the **Choose Files** button appears, select your downloaded **`samples.json`** file.

> *Running locally outside Google Colab?* Place `samples.json` in the same folder as this notebook and run the cell.

In [ ]:
import json
from pathlib import Path

SAMPLES_PATH = Path("samples.json")

try:
    from google.colab import files  # type: ignore
    print("Click 'Choose Files' below and select your downloaded samples.json file...")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file was uploaded. Re-run this cell and select samples.json.")
    SAMPLES_PATH = Path(next(iter(uploaded.keys())))
    print(f"Uploaded: {SAMPLES_PATH}")
except ImportError:
    print(f"Running locally; reading: {SAMPLES_PATH.resolve()}")

with open(SAMPLES_PATH, "r", encoding="utf-8") as f:
    dataset = json.load(f)

raw_samples = dataset.get("samples", [])
discovered_classes = sorted({str(s.get("label", "")).strip() for s in raw_samples if str(s.get("label", "")).strip()})
print(f"Success! Loaded {len(raw_samples)} gesture samples across {len(discovered_classes)} classes: {discovered_classes}")

## Step 2 – Inspect Your Dataset & Sample Balance

**What we're doing:** Counting how many samples you have for each gesture, checking whether both left and right hands are present, and plotting a quick bar chart.

**Why it matters:** Trying to collect roughly equal numbers of samples per gesture (aim for **30+**) keeps one gesture from overpowering the others. If your counts are uneven, don't worry—the notebook will automatically balance the training set in Step 4.

**What you need to do:** Just run this cell.

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt

label_counts = Counter(str(s.get("label", "")).strip() for s in raw_samples if str(s.get("label", "")).strip())
handedness_counts = Counter((s.get("handedness") or "Unknown") for s in raw_samples)

print(f"Found {len(label_counts)} gesture classes ({sum(label_counts.values())} samples total):")
for label, count in sorted(label_counts.items()):
    status = "Ready" if count >= 30 else "Tip: <30 samples (still usable, more is better)"
    print(f"  - {label}: {count} samples ({status})")
print("Handedness recorded:", dict(handedness_counts))

if len(label_counts) >= 2:
    counts_vals = list(label_counts.values())
    min_c, max_c = min(counts_vals), max(counts_vals)
    ratio = max_c / max(1, min_c)
    if ratio >= 1.6:
        under = [lbl for lbl, c in sorted(label_counts.items()) if c < max_c * 0.6]
        print(f"Note: Class counts are uneven ({min_c} to {max_c}, under-represented: {under}). Step 4 will balance the training data automatically.")

if label_counts:
    labels_sorted = sorted(label_counts.keys())
    counts_sorted = [label_counts[lbl] for lbl in labels_sorted]

    plt.figure(figsize=(max(8, len(labels_sorted) * 1.15), 4.0))
    bars = plt.bar(labels_sorted, counts_sorted, color="#f59e0b")
    plt.axhline(30, color="#34d399", linestyle="--", alpha=0.8, label="Recommended target (30)")
    plt.title(f"Samples Per Gesture ({len(labels_sorted)} Classes)")
    plt.xlabel("Gesture Name")
    plt.ylabel("Number of Samples")
    plt.xticks(rotation=25, ha="right")
    plt.legend()
    plt.grid(axis="y", alpha=0.25)
    for bar, count in zip(bars, counts_sorted):
        plt.text(bar.get_x() + bar.get_width() / 2, count + 0.4, str(count), ha="center", va="bottom")
    plt.tight_layout()
    plt.show()

## Step 3 – Prepare the Hand Landmark Features

**What we're doing:** Converting every raw hand snapshot into a consistent numeric fingerprint so the computer focuses on **finger shape**, not camera distance or screen position.

**Why we need this:**
- **Works with either hand:** Left-hand samples are mirrored into a shared Right-hand orientation so you can train with one hand and test with either.
- **Position & size neutral:** Every hand is centred at the wrist and scaled to a standard size.
- **Sharper shape clues:** We also measure finger joint bend angles and distances between fingertips so look-alike poses (such as **Open Palm** vs **Number 6**) are much easier to tell apart.

**What you need to do:** Nothing to edit—just run this cell.

In [ ]:
import math
from collections import Counter
import numpy as np

HAND_LANDMARK_COUNT = 21
HAND_FEATURE_COUNT = 63
CLASSIFICATION_GEOMETRIC_FEATURE_COUNT = 32
CLASSIFICATION_FEATURE_COUNT = HAND_FEATURE_COUNT + CLASSIFICATION_GEOMETRIC_FEATURE_COUNT
CANONICAL_HANDEDNESS = "Right"
CLASSIFICATION_COORD_WEIGHT_XY = 0.5
CLASSIFICATION_COORD_WEIGHT_Z = 0.3

HAND_BEND_ANGLE_TRIPLETS = [
    (0, 1, 2), (1, 2, 3), (2, 3, 4),
    (0, 5, 6), (5, 6, 7), (6, 7, 8),
    (0, 9, 10), (9, 10, 11), (10, 11, 12),
    (0, 13, 14), (13, 14, 15), (14, 15, 16),
    (0, 17, 18), (17, 18, 19), (18, 19, 20),
]
HAND_EXTENSION_PAIRS = [(2, 4), (5, 8), (9, 12), (13, 16), (17, 20)]
HAND_THUMB_GAP_TIPS = [8, 12, 16, 20]
HAND_ADJACENT_GAP_PAIRS = [(8, 12), (12, 16), (16, 20)]
HAND_PALM_DIST_TIPS = [4, 8, 12, 16, 20]

def normalize_handedness(handedness):
    """Exact Python match for normalizeHandedness() in src/vision/landmarkFeatures.js."""
    if not isinstance(handedness, str):
        return None
    clean = handedness.strip().lower()
    if clean == "left":
        return "Left"
    if clean == "right":
        return "Right"
    return None

def has_valid_hand_landmarks(landmarks):
    if not isinstance(landmarks, list) or len(landmarks) != HAND_LANDMARK_COUNT:
        return False
    for pt in landmarks:
        if not isinstance(pt, dict):
            return False
        try:
            x = float(pt["x"])
            y = float(pt["y"])
            z_raw = pt.get("z", 0.0)
            z = 0.0 if z_raw is None else float(z_raw)
        except (KeyError, TypeError, ValueError):
            return False
        if not (math.isfinite(x) and math.isfinite(y) and math.isfinite(z)):
            return False
    return True

def canonicalize_hand_landmarks(landmarks, handedness=None):
    """Exact Python match for canonicalizeHandLandmarks() in src/vision/landmarkFeatures.js."""
    if not has_valid_hand_landmarks(landmarks):
        return None
    mirror_x = normalize_handedness(handedness) == "Left"
    out = []
    for pt in landmarks:
        raw_x = float(pt["x"])
        z_raw = pt.get("z", 0.0)
        out.append({
            "x": (1.0 - raw_x) if mirror_x else raw_x,
            "y": float(pt["y"]),
            "z": 0.0 if z_raw is None else float(z_raw),
        })
    return out

def canonicalize_hand_feature_vector(features, handedness=None):
    """Exact Python match for canonicalizeHandFeatureVector() for legacy feature-only rows."""
    if not isinstance(features, list) or len(features) != HAND_FEATURE_COUNT:
        return None
    mirror_x = normalize_handedness(handedness) == "Left"
    out = []
    for idx, raw_val in enumerate(features):
        try:
            val = float(raw_val)
        except (TypeError, ValueError):
            return None
        if not math.isfinite(val):
            return None
        if mirror_x and (idx % 3 == 0):
            out.append(0.0 if val == 0.0 else -val)
        else:
            out.append(val)
    return out

def extract_hand_feature_vector(landmarks, handedness=None, canonicalize=True):
    """Exact Python match for extractHandFeatureVector() in src/vision/landmarkFeatures.js."""
    working = canonicalize_hand_landmarks(landmarks, handedness) if canonicalize else landmarks
    if not has_valid_hand_landmarks(working):
        return None

    wrist = working[0]
    wx = float(wrist["x"])
    wy = float(wrist["y"])
    wz = float(wrist.get("z", 0.0) or 0.0)

    shifted = []
    max_norm = 0.0

    for pt in working:
        dx = float(pt["x"]) - wx
        dy = float(pt["y"]) - wy
        dz = float(pt.get("z", 0.0) or 0.0) - wz
        if not (math.isfinite(dx) and math.isfinite(dy) and math.isfinite(dz)):
            return None
        shifted.extend((dx, dy, dz))
        norm = math.sqrt(dx * dx + dy * dy + dz * dz)
        if norm > max_norm:
            max_norm = norm

    if not math.isfinite(max_norm) or max_norm < 1e-5:
        return None

    inv_scale = 1.0 / max_norm
    return [v * inv_scale for v in shifted]

def extract_classification_features(features_63):
    """Exact Python match for extractClassificationFeatures() in src/vision/landmarkFeatures.js (95 floats)."""
    if not isinstance(features_63, (list, tuple)) or len(features_63) != HAND_FEATURE_COUNT:
        return None

    out = [0.0] * CLASSIFICATION_FEATURE_COUNT
    coords = [0.0] * HAND_FEATURE_COUNT
    for i in range(HAND_LANDMARK_COUNT):
        base = i * 3
        x = float(features_63[base])
        y = float(features_63[base + 1])
        z = float(features_63[base + 2])
        if not (math.isfinite(x) and math.isfinite(y) and math.isfinite(z)):
            return None
        coords[base] = x
        coords[base + 1] = y
        coords[base + 2] = z
        out[base] = x * CLASSIFICATION_COORD_WEIGHT_XY
        out[base + 1] = y * CLASSIFICATION_COORD_WEIGHT_XY
        out[base + 2] = z * CLASSIFICATION_COORD_WEIGHT_Z

    def _point_norm(idx):
        b = idx * 3
        px, py, pz = coords[b], coords[b + 1], coords[b + 2]
        return math.sqrt(px * px + py * py + pz * pz)

    def _pair_dist(a, b):
        ba = a * 3
        bb = b * 3
        dx = coords[ba] - coords[bb]
        dy = coords[ba + 1] - coords[bb + 1]
        dz = coords[ba + 2] - coords[bb + 2]
        return math.sqrt(dx * dx + dy * dy + dz * dz)

    palm_sum = _point_norm(5) + _point_norm(9) + _point_norm(13) + _point_norm(17)
    palm_scale = max(1e-4, palm_sum * 0.25)
    inv_two_palm = 1.0 / (2.0 * palm_scale)
    write_idx = HAND_FEATURE_COUNT

    for a, b, c in HAND_BEND_ANGLE_TRIPLETS:
        ba, bb, bc = a * 3, b * 3, c * 3
        v1x, v1y, v1z = coords[ba] - coords[bb], coords[ba + 1] - coords[bb + 1], coords[ba + 2] - coords[bb + 2]
        v2x, v2y, v2z = coords[bc] - coords[bb], coords[bc + 1] - coords[bb + 1], coords[bc + 2] - coords[bb + 2]
        n1 = math.sqrt(v1x * v1x + v1y * v1y + v1z * v1z)
        n2 = math.sqrt(v2x * v2x + v2y * v2y + v2z * v2z)
        bend = 0.0
        if n1 > 1e-6 and n2 > 1e-6:
            cos_raw = (v1x * v2x + v1y * v2y + v1z * v2z) / (n1 * n2)
            cos_clamped = max(-1.0, min(1.0, cos_raw))
            bend = 0.5 * (1.0 + cos_clamped)
        out[write_idx] = bend * 0.7
        write_idx += 1

    for mcp, tip in HAND_EXTENSION_PAIRS:
        ratio = (_point_norm(tip) - _point_norm(mcp)) * inv_two_palm
        clamped = max(-0.5, min(1.5, ratio))
        out[write_idx] = clamped * 0.75
        write_idx += 1

    for tip in HAND_THUMB_GAP_TIPS:
        dist = _pair_dist(4, tip) * inv_two_palm
        out[write_idx] = min(2.0, dist) * 0.7
        write_idx += 1

    for a, b in HAND_ADJACENT_GAP_PAIRS:
        dist = _pair_dist(a, b) * inv_two_palm
        out[write_idx] = min(2.0, dist) * 0.55
        write_idx += 1

    for tip in HAND_PALM_DIST_TIPS:
        dist = _pair_dist(9, tip) * inv_two_palm
        out[write_idx] = min(2.0, dist) * 0.65
        write_idx += 1

    return out

dataset_canonicalized = dataset.get("featureSpec", {}).get("handednessCanonicalization") is True
X_list = []
X_clf_list = []
y_list = []
meta_list = []
max_parity_diff = 0.0
recomputed_from_raw = 0
converted_left_features = 0
already_canonical = 0
unconvertible_missing_handedness = 0

for row in raw_samples:
    label = str(row.get("label", "")).strip()
    if not label or label == "None":
        continue

    handedness = normalize_handedness(row.get("handedness"))
    row_canon_flag = row.get("handednessCanonicalized")
    if not isinstance(row_canon_flag, bool):
        row_canon_flag = dataset_canonicalized

    raw_lm = row.get("rawLandmarks")
    existing_features = row.get("features")
    sample_canonicalized = False

    if has_valid_hand_landmarks(raw_lm):
        feats = extract_hand_feature_vector(raw_lm, handedness=handedness)
        if feats is None:
            continue
        if handedness is not None:
            sample_canonicalized = True
            if not row_canon_flag:
                recomputed_from_raw += 1
            else:
                already_canonical += 1
        else:
            sample_canonicalized = False
            unconvertible_missing_handedness += 1

        if isinstance(existing_features, list) and len(existing_features) == HAND_FEATURE_COUNT:
            expected_existing = [float(v) for v in existing_features]
            if row_canon_flag or handedness != "Left":
                diff = max(abs(a - b) for a, b in zip(feats, expected_existing))
                max_parity_diff = max(max_parity_diff, diff)
    elif isinstance(existing_features, list) and len(existing_features) == HAND_FEATURE_COUNT:
        parsed_feats = [float(v) for v in existing_features]
        if not all(math.isfinite(v) for v in parsed_feats):
            continue
        if row_canon_flag:
            feats = parsed_feats
            sample_canonicalized = True
            already_canonical += 1
        elif handedness == "Left":
            feats = canonicalize_hand_feature_vector(parsed_feats, "Left")
            sample_canonicalized = True
            converted_left_features += 1
        elif handedness == "Right":
            feats = parsed_feats
            sample_canonicalized = True
            already_canonical += 1
        else:
            feats = parsed_feats
            sample_canonicalized = False
            unconvertible_missing_handedness += 1
    else:
        continue

    feats_clf = extract_classification_features(feats)
    if feats_clf is None:
        continue

    X_list.append(feats)
    X_clf_list.append(feats_clf)
    y_list.append(label)
    meta_list.append({
        "label": label,
        "handedness": handedness,
        "handednessCanonicalized": sample_canonicalized,
        "features": feats,
    })

X = np.asarray(X_list, dtype=np.float32)
X_clf = np.asarray(X_clf_list, dtype=np.float32)
y = np.asarray(y_list)
clean_label_counts = Counter(y_list)
CLASSES = sorted(clean_label_counts.keys())

assert X.ndim == 2 and X.shape[1] == HAND_FEATURE_COUNT, f"Expected (*, 63), got {X.shape}"
print(f"Prepared canonical feature matrix X={X.shape}, {len(CLASSES)} classes={CLASSES}")
print(f"Ready! Built {len(X_clf_list)} clean 95-feature vectors across {len(CLASSES)} gestures.")
if unconvertible_missing_handedness > 0:
    print(f"Note: {unconvertible_missing_handedness} older sample(s) had no left/right tag and were kept as recorded.")
if len(CLASSES) < 2:
    print("Tip: Add at least 2 different gesture classes in the app before training.")

## Step 4 – Split Data Into Training & Practice-Test Sets

**What we're doing:** Dividing your samples into two groups:
- **80% Training set:** Examples the KNN model learns from.
- **20% Validation set:** Held-aside examples used like a mini-quiz to measure accuracy fairly.

If some gesture classes have fewer samples, this step also balances the training set automatically so every gesture gets equal voting power.

**What you need to do:** Just run this cell.

In [ ]:
from sklearn.model_selection import train_test_split

num_samples = len(X_clf)
num_classes = len(CLASSES)
min_class_count = min(clean_label_counts.values())

test_count = max(num_classes, int(round(num_samples * 0.2)))
use_stratify = min_class_count >= 2 and num_samples >= num_classes * 2 and (num_samples - test_count) >= num_classes
effective_test_size = (test_count / num_samples) if use_stratify else 0.2

X_train, X_val, y_train, y_val = train_test_split(
    X_clf,
    y,
    test_size=effective_test_size,
    random_state=42,
    stratify=y if use_stratify else None,
)

# Balance training split without leaking or duplicating validation samples
train_counts = Counter(y_train.tolist())
if len(train_counts) >= 2:
    target_train = max(train_counts.values())
    rng_bal = np.random.default_rng(42)
    bal_idx = []
    for cls in CLASSES:
        cls_pos = np.where(y_train == cls)[0]
        if len(cls_pos) == 0:
            continue
        if len(cls_pos) < target_train:
            extra = rng_bal.choice(cls_pos, size=target_train - len(cls_pos), replace=True)
            cls_pos = np.concatenate([cls_pos, extra])
        bal_idx.extend(cls_pos.tolist())
    X_train_bal = X_train[bal_idx]
    y_train_bal = y_train[bal_idx]
else:
    X_train_bal, y_train_bal = X_train, y_train

print(f"Split complete: {len(X_train)} training samples ({len(X_train_bal)} balanced) | {len(X_val)} validation samples.")

## Step 5 – Train Your KNN Model

**What we're doing:** Fitting a **K-Nearest Neighbours (KNN)** classifier on your balanced training samples.

**How KNN works:** Think of it like asking the **5 closest examples** (`K = 5`) in your dataset to vote whenever a new hand pose appears. Closer matches get a stronger vote.

**What you need to do:** Just run this cell.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

INITIAL_K = min(5, len(X_train_bal))
knn = KNeighborsClassifier(
    n_neighbors=INITIAL_K,
    weights="distance",
    metric="euclidean",
)
knn.fit(X_train_bal, y_train_bal)
print(f"Model trained! Using K={INITIAL_K} nearest neighbours across {len(CLASSES)} gesture classes.")

## Step 6 – Check How Well Your Model Performs

**What we're doing:** Testing the model on the held-out validation samples it has never seen and drawing a **Confusion Matrix**.

**How to read the chart:**
- Numbers along the **diagonal** (top-left to bottom-right) are correct predictions.
- Numbers outside the diagonal show gestures that confused the model. If two gestures mix up often, collecting 10–15 clearer samples in the app usually fixes it.

**What you need to do:** Just run this cell and inspect your score.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score

if len(X_val) > 0:
    y_pred = knn.predict(X_val)
    val_acc = float(accuracy_score(y_val, y_pred))
    macro_f1 = float(f1_score(y_val, y_pred, average="macro", zero_division=0))

    print(f"Validation Accuracy: {val_acc * 100:.1f}%  |  Macro F1 Score: {macro_f1:.3f}\n")
    print(classification_report(y_val, y_pred, labels=CLASSES, zero_division=0))

    fig_size = max(5.5, len(CLASSES) * 0.85)
    fig, ax = plt.subplots(figsize=(fig_size, fig_size * 0.85))
    ConfusionMatrixDisplay.from_predictions(
        y_val,
        y_pred,
        labels=CLASSES,
        cmap="YlOrBr",
        ax=ax,
        xticks_rotation=25,
    )
    plt.title(f"Confusion Matrix ({len(CLASSES)} Classes, K={INITIAL_K})")
    plt.tight_layout()
    plt.show()

    dists, _ = knn.kneighbors(X_val[:5], n_neighbors=1)
    vote_shares = knn.predict_proba(X_val[:5])
    print("Sample validation checks (NeighborShare = weighted vote agreement among top K neighbours):")
    for i in range(min(5, len(X_val))):
        print(f"  Actual={y_val[i]:14s} | Predicted={y_pred[i]:14s} | NeighborShare={vote_shares[i].max():.2f} | NearestDist={dists[i][0]:.3f}")
else:
    print("Not enough samples yet for a validation split—collect more samples in the app and re-run.")

### Optional Check – See How Different `K` Values Behave

**What we're doing:** Comparing accuracy when looking at `1, 3, 5, 7, or 9` nearest neighbours. Using **`K = 5`** works reliably for most workshop datasets because it smooths out minor hand wobble without blurring gestures together.

In [ ]:
candidate_ks = [k for k in [1, 3, 5, 7, 9] if k <= len(X_train_bal)]
k_scores = []

if len(X_val) > 0:
    for k_val in candidate_ks:
        clf = KNeighborsClassifier(n_neighbors=k_val, weights="distance", metric="euclidean")
        clf.fit(X_train_bal, y_train_bal)
        acc = float(accuracy_score(y_val, clf.predict(X_val)))
        k_scores.append(acc)
        print(f"K={k_val} -> Validation Accuracy: {acc * 100:.1f}%")

    if candidate_ks:
        plt.figure(figsize=(6, 3.2))
        plt.plot(candidate_ks, k_scores, marker="o", linewidth=2, color="#38bdf8")
        plt.title(f"Accuracy Across K ({len(CLASSES)} Classes)")
        plt.xlabel("K (Neighbours Checked)")
        plt.ylabel("Validation Accuracy")
        plt.ylim(0, 1.05)
        plt.xticks(candidate_ks)
        plt.grid(alpha=0.25)
        plt.tight_layout()
        plt.show()

SELECTED_K = min(5, len(X))
CONFIDENCE_THRESHOLD = 0.60
print(f"Using K={SELECTED_K} and minimum neighbour vote share={CONFIDENCE_THRESHOLD} for export.")

## Step 7 – Calibrate "Unknown" & "Uncertain" Rejection

**What we're doing:** Setting automatic distance limits so the app only accepts clear matches.

**Why it matters:** Without a distance limit, KNN would always force every random hand movement into whichever trained gesture happens to be closest. By measuring the distance gap between matching gestures (green bars) and different gestures (red bars), the app can safely say **`Unknown Gesture`** or **`Uncertain Gesture`** when you show something new.

**What you need to do:** Just run this cell (no edits needed).

In [ ]:
# Open-set distance & class-gap calibration using held-out validation samples
SHRINK_STRENGTH = 4.0
FALLBACK_GLOBAL_DISTANCE = 1.15
FALLBACK_MARGIN = 0.10
MIN_SIGMA_FLOOR = 0.06
JITTER_ALLOWANCE_FLOOR = 0.55
MIN_THRESHOLD_CLAMP = 0.40
MAX_THRESHOLD_CLAMP = 1.65

def _pairwise_min_dist(queries, gallery):
    if len(queries) == 0 or len(gallery) == 0:
        return np.array([], dtype=np.float64)
    diff = queries[:, None, :] - gallery[None, :, :]
    dists = np.sqrt(np.sum(diff * diff, axis=2))
    return np.min(dists, axis=1)

def _upper_inclass_estimate(in_dists):
    med = float(np.median(in_dists))
    mad = float(np.median(np.abs(in_dists - med)))
    sigma = max(1.4826 * mad, MIN_SIGMA_FLOOR)
    p95 = float(np.percentile(in_dists, 95))
    return max(p95 * 1.35, med + 3.0 * sigma), med, p95

in_class_by_label = {cls: [] for cls in CLASSES}
cross_class_by_label = {cls: [] for cls in CLASSES}
class_gaps = []
used_loo_fallback = False

for cls in CLASSES:
    val_mask = (y_val == cls) if len(X_val) > 0 else np.zeros(0, dtype=bool)
    train_same = X_train[y_train == cls]
    train_other = X_train[y_train != cls]

    if len(X_val) > 0 and np.any(val_mask) and len(train_same) > 0:
        val_cls = X_val[val_mask]
        in_d = _pairwise_min_dist(val_cls, train_same)
        in_class_by_label[cls].extend(in_d.tolist())
        if len(train_other) > 0:
            out_d = _pairwise_min_dist(val_cls, train_other)
            cross_class_by_label[cls].extend(out_d.tolist())
            class_gaps.extend(np.maximum(0.0, out_d - in_d).tolist())
    elif len(train_same) >= 2:
        used_loo_fallback = True
        diff = train_same[:, None, :] - train_same[None, :, :]
        dists = np.sqrt(np.sum(diff * diff, axis=2))
        np.fill_diagonal(dists, np.inf)
        in_d = np.min(dists, axis=1)
        in_class_by_label[cls].extend(in_d.tolist())
        if len(train_other) > 0:
            out_d = _pairwise_min_dist(train_same, train_other)
            cross_class_by_label[cls].extend(out_d.tolist())
            class_gaps.extend(np.maximum(0.0, out_d - in_d).tolist())

all_in_dists = np.asarray([d for cls in CLASSES for d in in_class_by_label[cls]], dtype=np.float64)
all_out_dists = np.asarray([d for cls in CLASSES for d in cross_class_by_label[cls]], dtype=np.float64)

if len(all_in_dists) > 0:
    in_upper_global, global_in_med, global_in_p95 = _upper_inclass_estimate(all_in_dists)
    if len(all_out_dists) > 0:
        cross_p10 = float(np.percentile(all_out_dists, 10))
        cross_cap = max(MIN_THRESHOLD_CLAMP, 0.80 * cross_p10)
        raw_global = min(max(in_upper_global, JITTER_ALLOWANCE_FLOOR), cross_cap)
    else:
        cross_p10 = None
        raw_global = max(in_upper_global, JITTER_ALLOWANCE_FLOOR)
    CALIBRATED_MAX_DISTANCE = round(float(min(max(raw_global, MIN_THRESHOLD_CLAMP), MAX_THRESHOLD_CLAMP)), 4)
    calibration_method = "held-out-val+loo-fallback" if used_loo_fallback else "held-out-validation"
else:
    global_in_med = None
    global_in_p95 = None
    cross_p10 = None
    CALIBRATED_MAX_DISTANCE = FALLBACK_GLOBAL_DISTANCE
    calibration_method = "fallback-default"

if len(class_gaps) > 0:
    gap_p10 = float(np.percentile(class_gaps, 10))
    CALIBRATED_MIN_DISTANCE_MARGIN = round(float(min(max(0.25 * gap_p10, 0.08), 0.22)), 4)
else:
    CALIBRATED_MIN_DISTANCE_MARGIN = FALLBACK_MARGIN

CLASS_DISTANCE_THRESHOLDS = {}
PER_CLASS_CALIBRATION_STATS = {}

for cls in CLASSES:
    c_in = np.asarray(in_class_by_label[cls], dtype=np.float64)
    c_out = np.asarray(cross_class_by_label[cls], dtype=np.float64)
    n_c = len(c_in)
    if n_c > 0:
        est_c, med_c, p95_c = _upper_inclass_estimate(c_in)
        if len(c_out) > 0:
            c_cross_p10 = float(np.percentile(c_out, 10))
            c_cross_min = float(np.min(c_out))
            c_cap = max(MIN_THRESHOLD_CLAMP, 0.82 * c_cross_p10)
            est_c = min(max(est_c, 0.45), c_cap)
        else:
            c_cross_p10 = None
            c_cross_min = None
            est_c = max(est_c, 0.45)

        shrink_w = n_c / (n_c + SHRINK_STRENGTH)
        shrunk_c = shrink_w * est_c + (1.0 - shrink_w) * CALIBRATED_MAX_DISTANCE
        thresh_c = round(float(min(max(shrunk_c, MIN_THRESHOLD_CLAMP), MAX_THRESHOLD_CLAMP)), 4)
        CLASS_DISTANCE_THRESHOLDS[cls] = thresh_c
        PER_CLASS_CALIBRATION_STATS[cls] = {
            "valCount": int(n_c),
            "inMedian": round(med_c, 4),
            "inP95": round(p95_c, 4),
            "crossMin": round(c_cross_min, 4) if c_cross_min is not None else None,
            "threshold": thresh_c,
        }
    else:
        CLASS_DISTANCE_THRESHOLDS[cls] = CALIBRATED_MAX_DISTANCE
        PER_CLASS_CALIBRATION_STATS[cls] = {
            "valCount": 0,
            "inMedian": None,
            "inP95": None,
            "crossMin": None,
            "threshold": CALIBRATED_MAX_DISTANCE,
        }

# Optional manual overrides (leave as None to use automatic calibration)
MANUAL_MAX_DISTANCE_OVERRIDE = None
MANUAL_MIN_MARGIN_OVERRIDE = None
MAX_DISTANCE_THRESHOLD = (
    float(MANUAL_MAX_DISTANCE_OVERRIDE)
    if MANUAL_MAX_DISTANCE_OVERRIDE is not None
    else CALIBRATED_MAX_DISTANCE
)
MIN_DISTANCE_MARGIN = (
    float(MANUAL_MIN_MARGIN_OVERRIDE)
    if MANUAL_MIN_MARGIN_OVERRIDE is not None
    else CALIBRATED_MIN_DISTANCE_MARGIN
)

print(f"Calibrated cutoff ({calibration_method}): max distance={MAX_DISTANCE_THRESHOLD:.3f} | min class gap={MIN_DISTANCE_MARGIN:.3f}")
for cls in CLASSES:
    st = PER_CLASS_CALIBRATION_STATS[cls]
    print(f"  - {cls:14s}: cutoff={st['threshold']:.3f} (typical match={st['inMedian']}, nearest other gesture={st['crossMin']})")

if len(all_in_dists) > 0:
    plt.figure(figsize=(7.2, 3.5))
    plt.hist(all_in_dists, bins=18, alpha=0.75, color="#34d399", label="Matching Gesture Distances")
    if len(all_out_dists) > 0:
        plt.hist(all_out_dists, bins=18, alpha=0.55, color="#f43f5e", label="Different Gesture Distances")
    plt.axvline(MAX_DISTANCE_THRESHOLD, color="#fbbf24", linestyle="--", linewidth=2, label=f"Unknown Cutoff ({MAX_DISTANCE_THRESHOLD:.2f})")
    plt.title("How Far Apart Are Your Gestures?")
    plt.xlabel("Feature Distance (Smaller = Closer Match)")
    plt.ylabel("Validation Samples")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

## Step 8 – Save & Download Your Trained Model (`knn-gesture-model.json`)

**What we're doing:** Exporting your trained KNN dataset, feature rules, and calibrated distance thresholds into **`knn-gesture-model.json`** so the web app can run it in real time.

**What you need to do:** Run this cell and wait for your browser to download **`knn-gesture-model.json`**.

> **Download didn't start?** Check if your browser blocked popups/downloads for Colab, or click the **Files** folder icon on Colab's left sidebar, right-click **`knn-gesture-model.json`**, and select **Download**.

In [ ]:
from datetime import datetime, timezone

# Refit evaluation classifier on balanced split to record accurate metrics for SELECTED_K
eval_k = min(SELECTED_K, len(X_train_bal))
eval_clf = KNeighborsClassifier(n_neighbors=eval_k, weights="distance", metric="euclidean")
eval_clf.fit(X_train_bal, y_train_bal)
if len(X_val) > 0:
    eval_pred = eval_clf.predict(X_val)
    final_val_acc = float(accuracy_score(y_val, eval_pred))
    final_macro_f1 = float(f1_score(y_val, eval_pred, average="macro", zero_division=0))
else:
    final_val_acc = 1.0
    final_macro_f1 = 1.0

model_payload = {
    "schemaVersion": 4,
    "modelType": "knn-hand-landmarks",
    "createdAt": datetime.now(timezone.utc).isoformat(),
    "featureSpec": {
        "featureCount": 63,
        "classificationFeatureCount": CLASSIFICATION_FEATURE_COUNT,
        "landmarkCount": 21,
        "dimensions": ["x", "y", "z"],
        "coordinateSource": "hand.landmarks",
        "coordinateSpace": "raw-landmarks-wrist-centered-maxnorm",
        "handednessCanonicalization": True,
        "canonicalHand": CANONICAL_HANDEDNESS,
    },
    "hyperparams": {
        "k": int(SELECTED_K),
        "metric": "euclidean",
        "weighting": "distance",
        "confidenceThreshold": float(CONFIDENCE_THRESHOLD),
        "maxDistanceThreshold": float(MAX_DISTANCE_THRESHOLD) if MAX_DISTANCE_THRESHOLD is not None else None,
        "minDistanceMargin": float(MIN_DISTANCE_MARGIN) if MIN_DISTANCE_MARGIN is not None else None,
        "classDistanceThresholds": CLASS_DISTANCE_THRESHOLDS,
    },
    "calibration": {
        "method": calibration_method,
        "shrinkStrength": float(SHRINK_STRENGTH),
        "validationSampleCount": int(len(X_val)),
        "inClassMedian": round(global_in_med, 4) if global_in_med is not None else None,
        "inClassP95": round(global_in_p95, 4) if global_in_p95 is not None else None,
        "crossClassP10": round(cross_p10, 4) if cross_p10 is not None else None,
        "globalMaxDistanceThreshold": float(CALIBRATED_MAX_DISTANCE),
        "minDistanceMargin": float(CALIBRATED_MIN_DISTANCE_MARGIN),
        "perClassStats": PER_CLASS_CALIBRATION_STATS,
    },
    "classes": CLASSES,
    "classCounts": {cls: int(clean_label_counts[cls]) for cls in CLASSES},
    "sampleCount": len(meta_list),
    "metrics": {
        "validationAccuracy": round(final_val_acc, 4),
        "macroF1": round(final_macro_f1, 4),
        "trainCount": int(len(X_train)),
        "valCount": int(len(X_val)),
    },
    "samples": [
        {
            "label": item["label"],
            "handedness": item["handedness"],
            "handednessCanonicalized": bool(item["handednessCanonicalized"]),
            "features": [round(float(v), 6) for v in item["features"]],
        }
        for item in meta_list
    ],
}

OUTPUT_PATH = Path("knn-gesture-model.json")
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(model_payload, f, indent=2)

print(f"Success! Exported '{OUTPUT_PATH}' ({len(meta_list)} samples, {len(CLASSES)} gestures, validation accuracy={final_val_acc:.1%})")

try:
    from google.colab import files  # type: ignore
    files.download(str(OUTPUT_PATH))
    print("Download started! Look for knn-gesture-model.json in your Downloads folder.")
except ImportError:
    print(f"Saved locally at: {OUTPUT_PATH.resolve()}")

## All Done! Bring Your Model Back to the App

1. Return to the **HELLO, VISION!** web app and open **Custom Gestures -> 2. Train Model**.
2. Under **Step 3 — Import & Test Your Model**, click **Import Model JSON** and select **`knn-gesture-model.json`**.
3. Open **3. Test & Map Actions**, make sure **Custom KNN (Colab)** is selected, and try your gestures in front of your webcam:
   - Test with both your **left and right hand**.
   - Try an unfamiliar hand shape to verify it safely reports **`Unknown Gesture`**.
4. Assign your custom gestures to actions in **Magic Canvas** and **Particle Universe**!